# 03. LightGBM 학습 및 예측

## 🎯 목적

02단계에서 생성한 통합 데이터셋으로 **전체 통합 LightGBM 모델** 학습

## 📋 핵심 설계

### ✅ Target: 로그 종가 절대값
```python
target_log_close = log(close)
```
- t일 features → t+1일 log_close 예측
- 절대값 예측이므로 종목 간 가격 수준 차이 반영

### ✅ Feature: ticker 제외
- **제외 이유**: 신규 종목 예측 불가, 차원 폭발
- **대체 수단**: Meta features (liquidity_score, risk_composite)
- **장점**: 2,900개 → 12개 features로 일반화

### ✅ 학습 방식: 전체 통합 모델
- 단일 모델로 모든 종목 처리
- 종목 간 공통 패턴 학습
- 신규 상장 종목도 즉시 예측 가능

## 🔄 데이터 흐름

```
data/03_processed/dataset.parquet
    ↓ [load]
  + Target 생성: log(close)
    ↓ [walk_forward_split]
  train / valid (4 folds) / test
    ↓ [LightGBM 학습]
  Unified Model (NO ticker feature)
    ↓ [save]
data/04_models/*.pkl
data/05_results/*.csv
```

## 🔧 Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings

from src.models.lightgbm_model import LightGBMModel
from src.modeling.trainer import WalkForwardTrainer
from src.models.artifact import save_model_artifact

warnings.filterwarnings('ignore')
print("✅ Modules loaded")

In [ ]:
# ==================== 설정 ====================

REFERENCE_DATE = "20260112"
DATASET_PATH = Path(f"data/03_processed/dataset_{REFERENCE_DATE}.parquet")
MODEL_DIR = Path("data/04_models")
RESULT_DIR = Path("data/05_results")

# 디렉토리 생성
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Target 설정
TARGET_COL = "target_log_close"

# Walk-Forward 설정
TRAIN_END = "2025-01-10"  # 학습 종료일
VALID_WINDOW = 60         # 검증 구간 (거래일)
TEST_WINDOW = 63          # 테스트 구간
NUM_VALID = 3             # 검증 구간 개수

# LightGBM 하이퍼파라미터
LGBM_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
}

print(f"📁 Dataset: {DATASET_PATH}")
print(f"🎯 Target: log(close) 절대값 (t일 → t+1일 예측)")
print(f"📅 Train end: {TRAIN_END}")
print(f"📊 Valid: {NUM_VALID} folds x {VALID_WINDOW} days")
print(f"🧪 Test: {TEST_WINDOW} days")

## 1️⃣ 데이터 로드 및 Target 생성

In [ ]:
# ==================== 데이터 로드 ====================

print("📥 Loading dataset...")
df = pd.read_parquet(DATASET_PATH)

print(f"   - Shape: {df.shape}")
print(f"   - Tickers: {df['ticker'].nunique():,}")
print(f"   - Period: {df['date'].min()} ~ {df['date'].max()}")

# Feature 컬럼 자동 추출 (ticker 제외)
feature_cols = [c for c in df.columns if c.startswith('feature_')]

# Meta features 추가 (종목 특성 표현)
meta_features = ['liquidity_score', 'risk_composite']
available_meta = [c for c in meta_features if c in df.columns]
feature_cols.extend(available_meta)

print(f"\n✅ Features: {len(feature_cols)}개 (NO ticker)")
print(f"   Technical: {[c for c in feature_cols if 'feature_tech' in c][:3]}...")
print(f"   Meta: {available_meta}")
print(f"\n💡 Meta features가 종목 특성을 대체 → 신규 종목 예측 가능")

In [ ]:
# ==================== Target 생성 ====================

print(f"\n🎯 Creating Target: log(close)...")

# 로그 종가 계산
df[TARGET_COL] = np.where(df['close'] > 0, np.log(df['close']), 0)

print(f"   Target column: {TARGET_COL}")
print(f"   Meaning: t일 features → t+1일 log_close 예측")

# Target 분포 확인
print(f"\n📊 Target Distribution:")
print(df[TARGET_COL].describe())

# NaN 체크 (close=0이면 log 불가)
nan_count = df[TARGET_COL].isna().sum()
if nan_count > 0:
    print(f"\n⚠️  Target NaN: {nan_count} ({nan_count/len(df)*100:.3f}%)")
    print("   → close=0인 경우 발생 (거래정지 등)")
    df = df.dropna(subset=[TARGET_COL])
    print(f"   → 제거 후: {len(df):,} rows")

## 2️⃣ 모델 학습 (Walk-Forward)

In [ ]:
# ==================== 모델 초기화 ====================

print("🔧 Initializing LightGBM Model...")

model = LightGBMModel(
    model_version="v1_unified_no_ticker",
    params=LGBM_PARAMS,
    feature_list=feature_cols,
    categorical_features=[],  # ticker 제외
    task="regression"
)

print(f"   Model: {model.model_name} v{model.model_version}")
print(f"   Features: {len(feature_cols)} (NO ticker)")
print(f"   → 신규 종목도 meta features만 있으면 예측 가능")

In [ ]:
# ==================== Trainer 실행 ====================

print("\n" + "="*65)
print("🚀 Starting Walk-Forward Training...")
print("="*65)

trainer = WalkForwardTrainer(
    model=model,
    feature_cols=feature_cols,
    target_col=TARGET_COL,
    date_col='date',
    categorical_features=[]
)

results = trainer.run(
    df=df,
    train_end=TRAIN_END,
    valid_window_days=VALID_WINDOW,
    test_window_days=TEST_WINDOW,
    num_valid=NUM_VALID,
    fit_kwargs={
        'num_boost_round': 1000,
        'callbacks': [
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    }
)

print("\n✅ Training completed!")

## 3️⃣ 성능 평가

In [ ]:
# ==================== 성능 지표 출력 ====================

print("\n" + "="*65)
print("📊 Performance Summary")
print("="*65)

# Train
print("\n[Train Set]")
for k, v in results['train_metrics'].items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# Valid (평균)
print("\n[Validation Set - Average]")
valid_avg = {}
for key in results['valid_metrics'][0].keys():
    if key != 'samples':
        valid_avg[key] = np.mean([v[key] for v in results['valid_metrics']])
    else:
        valid_avg[key] = sum([v[key] for v in results['valid_metrics']])

for k, v in valid_avg.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# Test
print("\n[Test Set]")
for k, v in results['test_metrics'].items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# 과적합 지표
print("\n[Overfitting Check]")
rmse_gap = results['test_metrics']['rmse'] - valid_avg['rmse']
print(f"  RMSE gap (test - valid): {rmse_gap:+.6f}")
if rmse_gap > 0.02:
    print("  ⚠️  과적합 의심")
else:
    print("  ✅ 일반화 양호")

In [ ]:
# ==================== Feature Importance ====================

print("\n📈 Feature Importance (Top 10):")

meta = model.get_meta()
importance_dict = meta.get('feature_importance', {})

if importance_dict:
    importance_df = pd.DataFrame([
        {'feature': k, 'importance': v} 
        for k, v in importance_dict.items()
    ]).sort_values('importance', ascending=False)
    
    print(importance_df.head(10).to_string(index=False))
    
    # Meta features 중요도 체크
    meta_importance = importance_df[importance_df['feature'].isin(available_meta)]
    if not meta_importance.empty:
        print(f"\n💡 Meta Features Importance:")
        print(meta_importance.to_string(index=False))
        print("   → 종목 특성을 얼마나 잘 포착했는지 확인")
    
    # 저장
    importance_path = RESULT_DIR / f"feature_importance_{REFERENCE_DATE}.csv"
    importance_df.to_csv(importance_path, index=False)
    print(f"\n💾 Saved: {importance_path}")
else:
    print("  (Feature importance not available)")

## 4️⃣ 모델 및 결과 저장

In [ ]:
# ==================== 모델 저장 ====================

print("\n💾 Saving model artifact...")

artifact_path = save_model_artifact(
    model_name="lightgbm",
    model_version="v1_unified_no_ticker",
    model_object=model,
    metadata={
        "feature_list": feature_cols,
        "training_period": f"~{TRAIN_END}",
        "hyperparameters": LGBM_PARAMS,
        "data_version": REFERENCE_DATE,
        "target": TARGET_COL,
        "test_rmse": results['test_metrics']['rmse'],
        "test_r2": results['test_metrics']['r2'],
        "note": "No ticker feature - can predict new stocks with meta features",
    },
    model_dir=MODEL_DIR
)

print(f"✅ Model saved: {artifact_path}")
print(f"\n💡 이 모델의 특징:")
print(f"   1. 신규 상장 종목도 예측 가능 (ticker 불필요)")
print(f"   2. Meta features로 종목 특성 표현")
print(f"   3. 종목 수가 증가해도 재학습 불필요")

In [ ]:
# ==================== 예측 결과 저장 ====================

print("\n💾 Saving predictions...")

pred_df = results['test_predictions']
pred_path = RESULT_DIR / f"predictions_{REFERENCE_DATE}.csv"
pred_df.to_csv(pred_path, index=False, encoding='utf-8-sig')

print(f"✅ Predictions saved: {pred_path}")
print(f"   Shape: {pred_df.shape}")
print(f"\n📋 Sample:")
print(pred_df.head(10))

In [ ]:
# ==================== 지표 저장 ====================

print("\n💾 Saving metrics summary...")

metrics_summary = {
    'reference_date': REFERENCE_DATE,
    'train_end': TRAIN_END,
    'target': TARGET_COL,
    'num_features': len(feature_cols),
    'ticker_as_feature': False,
    **{f'train_{k}': v for k, v in results['train_metrics'].items()},
    **{f'valid_avg_{k}': v for k, v in valid_avg.items()},
    **{f'test_{k}': v for k, v in results['test_metrics'].items()},
}

metrics_df = pd.DataFrame([metrics_summary])
metrics_path = RESULT_DIR / f"metrics_summary_{REFERENCE_DATE}.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"✅ Metrics saved: {metrics_path}")

## ✅ 완료!

### 생성된 파일
- 모델: `data/04_models/lightgbm/*.pkl`
- 예측: `data/05_results/predictions_*.csv`
- 지표: `data/05_results/metrics_summary_*.csv`
- Feature Importance: `data/05_results/feature_importance_*.csv`

### 핵심 설계 특징

✅ **Ticker를 feature로 사용하지 않음**
- 신규 종목 예측 가능
- 차원 폭발 문제 없음
- Meta features (liquidity, risk)로 종목 특성 표현

✅ **로그 종가 절대값 예측**
- t일 features → t+1일 log(close)
- 종목 간 가격 수준 차이 반영

✅ **전체 통합 모델**
- 단일 모델로 모든 종목 처리
- 종목 간 공통 패턴 학습

### 모델 사용 예시
```python
from src.models.lightgbm_model import LightGBMModel

# 신규 종목 데이터 준비
new_stock = pd.DataFrame({
    'feature_ma_5': [...],
    'liquidity_score': [...],  # Meta feature
    'risk_composite': [...]     # Meta feature
    # ticker 필요 없음!
})

# 예측
loaded_model = LightGBMModel.load('data/04_models/lightgbm/*.pkl')
predictions = loaded_model.predict(new_stock)
```

### 다음 단계
- **트랙 D**: Universe 선정 및 시그널 생성
- **백테스트**: 실제 매매 시뮬레이션